In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


In [2]:
df = pd.read_csv("Task 3 and 4_Loan_Data.csv")

# Drop ID column
df = df.drop(columns=["customer_id"])

X = df.drop(columns=["default"])
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


In [3]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
pd_model = LogisticRegression(max_iter=1000)
pd_model.fit(X_train_scaled, y_train)

pd_train = pd_model.predict_proba(X_train_scaled)[:, 1]
pd_test = pd_model.predict_proba(X_test_scaled)[:, 1]

print("Train AUC:", roc_auc_score(y_train, pd_train))
print("Test AUC:", roc_auc_score(y_test, pd_test))

Train AUC: 0.9999832259796028
Test AUC: 0.9999896829344681


In [7]:
def expected_loss(
    borrower_row: pd.DataFrame,
    model,
    scaler,
    recovery_rate=0.10
):
    scaled = scaler.transform(borrower_row)
    pd_estimate = model.predict_proba(scaled)[0, 1]

    lgd = 1 - recovery_rate
    loan_exposure = borrower_row["loan_amt_outstanding"].values[0]

    el = pd_estimate * loan_exposure * lgd

    return {
        "PD": pd_estimate,
        "Expected_Loss": el
    }


In [8]:
sample_borrower = pd.DataFrame([{
    "credit_lines_outstanding": 3,
    "loan_amt_outstanding": 6000,
    "total_debt_outstanding": 12000,
    "income": 42000,
    "years_employed": 2,
    "fico_score": 580
}])

result = expected_loss(
    borrower_row=sample_borrower,
    model=pd_model,
    scaler=scaler
)

print(result)

{'PD': np.float64(0.9952483777545378), 'Expected_Loss': np.float64(5374.341239874504)}


In [9]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": pd_model.coef_[0]
}).sort_values(by="Coefficient")

coef_df

,Feature,Coefficient
4,years_employed,-2.750583
3,income,-2.169045
5,fico_score,-1.094128
1,loan_amt_outstanding,0.116525
2,total_debt_outstanding,3.540072
0,credit_lines_outstanding,8.404819


In [10]:
coef_df["Odds_Ratio"] = np.exp(coef_df["Coefficient"])
coef_df

,Feature,Coefficient,Odds_Ratio
4,years_employed,-2.750583,0.063891
3,income,-2.169045,0.114287
5,fico_score,-1.094128,0.334832
1,loan_amt_outstanding,0.116525,1.123585
2,total_debt_outstanding,3.540072,34.469412
0,credit_lines_outstanding,8.404819,4468.550436


In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

tree_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=50,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_pd = tree_model.predict_proba(X_test)[:, 1]
print("Decision Tree AUC:", roc_auc_score(y_test, tree_pd))

Decision Tree AUC: 0.9994870944564196
